## SummarizationMiddleware中间件
对历史消息列表进行 摘要&总结 ，达到 `压缩上下文` 的效果。

原理：在 `达到触发条件` 时，调用大模型对历史消息进行摘要， 将摘要的结果作为`HumanMessage` ，
放到消息列表最开始的位置。

### 参数说明
##### 参数1:model 用于摘要的模型
可以是模型名称也可以是模型对象，如果传递的是模型名称，底层会调用`init_chat_model`初始化模型。

##### trigger 摘要触发条件
是一个列表，每个元素对应一个条件。当`任意条件满足`时，触发摘要。
1. tokens ：token的数量，历史token的累计数量达到该值触发摘要。
2. messages ：历史消息数量，历史消息条数达到该值触发摘要。
3. fraction ：上下文长度比例。历史token的累计数量达到模型的 `max_input_tokens*fraction` 触
发摘要
    如果条件包含 fraction ，要求模型的profile包含 max_input_tokens ，Deepseek模型的profile为空，此时需要手动添加该配置项。Deepseek-V3.2的上下文长度为128K。

##### keep 摘要时保留的原始消息
支持三种条件，但和trigger不同，keep同一时间只接收一种条件。
1. tokens ：摘要时保留的token数量。
2. messages ：摘要时保留的历史消息条数。
3. fraction ：摘要时保留 max_input_tokens*fraction 个token。

##### token_counter 统计token数量的函数
默认使用LangChain提供的 count_tokens_approximately ，一般不用更改。

##### summary_prompt 摘要时的自定义提示词
该提示词需要包含 {messages} 占位符，使得历史消息列表可以被插入。不指定则使用内置提示词。

##### trim_token_to_summarize —摘要时历史消息的最大token数
如果历史消息token数大于该值，则会被裁剪。默认为" 4000 "。
如果trigger用token作为度量，调大触发阈值时，当前配置项应相应调整，否则会丢失信息。

In [ ]:
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)


In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，你是傻逼，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

agent = create_agent(
    model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[ # 触发条件
                ("tokens", 100), # 历史token的累计数量达到该值触发摘要。
                ("messages", 6), # 历史消息数量，历史消息条数达到该值触发摘要。
                ("fraction", 0.001) # 上下文长度比例。历史token的累计数量达到模型的 `max_input_tokens*fraction` 触发摘要
            ],
            keep=("messages", 2) # 摘要时保留的原始消息
        )
    ]
)

response = agent.invoke({"messages": messages})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
Establish initial rapport and exchange identities. The user initiated contact to introduce themselves and ask for the assistant's identity.

## SUMMARY
- User introduced themselves as "老王" (Lao Wang).
- Assistant identified itself as "小王" (Xiao Wang).
- User acknowledged the introduction and expressed positive rapport.
- No technical tasks, strategic decisions, rejected options, or workflow steps were established. The conversation remains at the introductory/greeting phase.

## ARTIFACTS
None

## NEXT STEPS
Await the user's first substantive request, query, or task to begin active assistance.
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message =========================